In [1]:
# 수학 체인
# 사용자 질문 >> 수학 함수 호출 체인

!pip install langchain
!pip install openai
!pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 3.8 MB/s eta 0:00:00


In [2]:
import os
import openai

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('kosa')

In [3]:
# 수학공식 푸는 패키지 numexpr

import numexpr

In [4]:
numexpr.evaluate('15**0.3432')

array(2.53299609)

In [5]:
numexpr.evaluate('15**2')

array(225, dtype=int32)

In [6]:
# 내가 원하는 거
# 자연어 질문 >> 수학공식 생성 >> numexpr >> 답변

from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate, AIMessagePromptTemplate


# PromptTemplate : 대화의 전반적인 흐름 제어하는 템플릿
# SystemMessagePromptTemplate : 시스템 메시지만 포함하는 템플릿
# HumanMessagePromptTemplate: 사용자 메시지만 포함하는 템플릿

In [7]:
# numexpr.evaluate("수학공식")
# 자연어 >> llm >> 수학공식 >> parsing >> numexpr.evalute

# 출력형식
'''
```text
13**0.3432
```
'''
# ai 가 정해진 질문 이해, 답변 role (역할) 부여

system_content_template = """You are a helpful assistant that make an only ARGUMENT for numexpr.evaluate according to user query such as
```text
13**0.3432
```
when user query is 'What is 13 raised to the .3432 power?'

Please response as code block using text back quote like
```text
13**0.3432
```
without other description."""

In [8]:
# 실제 사용자가 어떤 형식으로 input 전달할지 정의
human_content_template = "{user_input}"

# 템플릿 정의
chat_prompt_template = ChatPromptTemplate.from_messages([
  ("system", system_content_template),
  ('human', human_content_template)
])

chat_model = ChatOpenAI(temperature=0)

chain = chat_prompt_template | chat_model

chain.invoke({"user_input":"What is 15 raised to the .3432 power?"})

AIMessage(content='```text\n15**0.3432\n```', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 101, 'total_tokens': 112, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DkUoe401LYcIV7WVbqYexNZV3DBPQ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e6eb6-812d-7d62-8169-37f928f39d69-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 101, 'output_tokens': 11, 'total_tokens': 112, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [10]:
# 모델의 출력을 특정 형식으로 구문 분석 >> 필요에 따라 추가적인 처리를 수행하는 데 사용
from langchain_core.output_parsers import BaseOutputParser

# ```text\n15**0.3432\n``` >> "15**0.3432"

class CodeOutputParser(BaseOutputParser):
  def parse(self, text: str):
    # text 변수 문자열(str)  전달되어야 함 명시
    value = text.strip().split("```")
    # 공백 제거, ``` 기준 구분 (분리)
    print(value)
    return value[1].split("\n")[1]


chain = chat_prompt_template | chat_model | CodeOutputParser()

output = chain.invoke({"user_input":"What is 15 raised to the .3432 power?"})
print(output)

['', "text\nnumexpr.evaluate('15**0.3432')\n", '']
numexpr.evaluate('15**0.3432')


In [11]:
chain = chat_prompt_template | chat_model | CodeOutputParser() | numexpr.evaluate

output = chain.invoke({"user_input":"What is 15 raised to the .3432 power?"})
print(output)

['', 'text\n15**0.3432\n', '']
2.5329960941138205


In [12]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda

chain = ({'user_input': RunnablePassthrough()}
        | chat_prompt_template
        | chat_model
        | CodeOutputParser()
        | numexpr.evaluate

         )

output = chain.invoke("What is 15 raised to the .3432 power?")
print(output)

['', 'text\n15**0.3432\n', '']
2.5329960941138205


In [ ]:
# eos